# 04 · 向量数学：点积 / L2 范数 / 余弦相似度 / softmax / 交叉熵

> **学习目标**：手算 + NumPy 验证后续 RAG / Attention / Loss 里**必出现**的 5 个数学量。
>
> **预备**：高中向量；做过 02 号 notebook。
>
> **为什么重要**：所有「相似度匹配」「分类输出」「loss 计算」都是这 5 个量的组合。手算一遍，以后看公式不再发懵。

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## 1. 点积（Dot Product）

$$\mathbf{a} \cdot \mathbf{b} = \sum_i a_i b_i$$

**几何含义**：`|a| · |b| · cos(夹角)`。两个方向越一致，点积越大。

**LLM 场景**：attention 里 Q 和 K 算分数、向量库做相似度检索 —— 全是点积。

**手算示例**：`a = [1, 2, 3]`, `b = [2, 0, 1]`
- `a · b = 1×2 + 2×0 + 3×1 = 2 + 0 + 3 = 5`

In [ ]:
a = np.array([1, 2, 3])
b = np.array([2, 0, 1])

# 4 种等价写法
print('a @ b           =', a @ b)
print('np.dot(a, b)    =', np.dot(a, b))
print('(a * b).sum()   =', (a * b).sum())
print('np.inner(a, b)  =', np.inner(a, b))

assert int(a @ b) == 5, '手算与代码不一致，检查推导！'

## 2. L2 范数（向量长度）

$$\lVert \mathbf{a} \rVert_2 = \sqrt{\sum_i a_i^2}$$

**手算示例**：`a = [1, 2, 3]`
- `||a|| = sqrt(1 + 4 + 9) = sqrt(14) ≈ 3.7417`

**LLM 场景**：embedding 归一化（除以范数）后做相似度 = 直接点积 = 余弦相似度。

In [ ]:
norm_a = np.linalg.norm(a)
norm_b = np.linalg.norm(b)
print(f'||a|| = {norm_a:.4f}   期望 sqrt(14) = {np.sqrt(14):.4f}')
print(f'||b|| = {norm_b:.4f}   期望 sqrt(5)  = {np.sqrt(5):.4f}')

# 归一化：a / ||a||  得到方向（单位向量）
a_unit = a / norm_a
print('a 归一化后:', a_unit)
print('归一化后的范数应为 1:', np.linalg.norm(a_unit))

## 3. 余弦相似度（Cosine Similarity）

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert \cdot \lVert \mathbf{b} \rVert}$$

**范围**：[-1, 1]。1 = 同方向（最相似），0 = 垂直（无关），-1 = 反方向。

**手算示例**：
- `cos(a, b) = 5 / (sqrt(14) · sqrt(5)) = 5 / sqrt(70) ≈ 5 / 8.3666 ≈ 0.5976`

In [ ]:
def cosine_sim(x, y):
    return (x @ y) / (np.linalg.norm(x) * np.linalg.norm(y))

cs = cosine_sim(a, b)
print(f'cos(a, b) = {cs:.4f}   期望 ≈ 0.5976')

# 验证：归一化后两个单位向量的点积 = 余弦相似度
a_u = a / np.linalg.norm(a)
b_u = b / np.linalg.norm(b)
print(f'a_u @ b_u  = {a_u @ b_u:.4f}   <- 与上面一致')

In [ ]:
# 批量版：query 一个向量、与一批 doc 算相似度 —— 这就是向量库检索的核心
query = np.array([1.0, 2.0, 3.0])
docs = np.array([
    [1, 2, 3],          # doc0 与 query 完全一致 -> sim = 1
    [2, 4, 6],           # doc1 是 query 的 2 倍方向相同 -> sim = 1
    [-1, -2, -3],        # doc2 反方向 -> sim = -1
    [3, 2, 1],           # doc3 互换 -> sim 中等
    [0, 0, 1],           # doc4 只有 z 分量
])

q_u = query / np.linalg.norm(query)
d_u = docs / np.linalg.norm(docs, axis=1, keepdims=True)   # 广播：每行除自己的范数
sims = d_u @ q_u                # (5, 3) @ (3,) = (5,)

for i, s in enumerate(sims):
    print(f'doc{i} {docs[i]!s:15} sim = {s:+.4f}')
print('\ntop-2 idx (从高到低):', np.argsort(-sims)[:2])

## 4. Softmax — 把一堆数变成概率

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

**性质**：输出每个 ∈ (0, 1)，加起来 = 1。大的输入获得指数级大的概率。

**手算示例**：`x = [1, 2, 3]`
- `e^1 ≈ 2.718, e^2 ≈ 7.389, e^3 ≈ 20.086`
- 和 ≈ `30.19`
- softmax ≈ `[2.72/30.19, 7.39/30.19, 20.09/30.19] ≈ [0.0900, 0.2447, 0.6652]`

In [ ]:
def softmax(x, axis=-1):
    # 减最大值是数值稳定技巧 —— 见 02 号 notebook
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

x = np.array([1.0, 2.0, 3.0])
p = softmax(x)
print('softmax([1,2,3]) =', p, '期望 ≈ [0.090, 0.245, 0.665]')
print('概率和 =', p.sum())

In [ ]:
# 温度的影响 —— 这是 LLM 采样的核心
logits = np.array([1.0, 2.0, 3.0])
for T in [0.1, 0.5, 1.0, 2.0, 10.0]:
    p = softmax(logits / T)
    print(f'T={T:5.1f}  softmax = {p}')
print('\n→ 温度 ↓: 分布更尖锐，几乎只选最大；温度 ↑: 分布趋平，更随机')

## 5. 交叉熵 Loss — 模型预测 vs 真实标签

$$\text{CE}(p, y) = -\log(p_y)$$

其中 `p` 是 softmax 输出的概率分布，`y` 是真实类别。**模型对正确类别预测的概率越接近 1，loss 越接近 0；越接近 0，loss 越大。**

**手算示例**：模型输出 logits `[1, 2, 3]`，真实标签是 2（第 3 类）。
- softmax ≈ `[0.090, 0.245, 0.665]`
- 取真实类别概率：`p[2] ≈ 0.665`
- CE = `-log(0.665) ≈ 0.408`

In [ ]:
logits = np.array([1.0, 2.0, 3.0])
true_label = 2

p = softmax(logits)
ce = -np.log(p[true_label])
print(f'CE loss = {ce:.4f}   期望 ≈ 0.408')

# 与 PyTorch 对比
import torch, torch.nn.functional as F
logits_t = torch.tensor([[1.0, 2.0, 3.0]])
target_t = torch.tensor([2])
ce_torch = F.cross_entropy(logits_t, target_t)
print(f'torch.cross_entropy = {ce_torch.item():.4f}   <- 应该一致')

In [ ]:
# 训练直觉：loss 随预测准确度变化
import matplotlib.pyplot as plt

p_true = np.linspace(0.01, 0.99, 100)        # 模型对真实类别的预测概率
loss = -np.log(p_true)

plt.figure(figsize=(7, 4))
plt.plot(p_true, loss)
plt.axhline(np.log(3), ls='--', color='gray', label='-log(1/3) ≈ 1.10  (3 类随机猜)')
plt.xlabel('模型对正确类别的预测概率 p_true')
plt.ylabel('cross entropy loss')
plt.title('CE loss = -log(p_true)')
plt.legend(); plt.grid(True); plt.show()
print('p_true=0.99 -> loss', -np.log(0.99))
print('p_true=0.50 -> loss', -np.log(0.50))
print('p_true=0.01 -> loss', -np.log(0.01))

## 深入思考

1. **为什么 RAG 检索几乎全用余弦相似度，不用 L2 距离？**
   - embedding 模型常输出已归一化或 magnitude 接近的向量；余弦只关心方向，对范数不敏感，更稳。
   - 但严格说，归一化后的 L2² = 2 - 2·cos，**等价**。
2. **如果两个向量做内积都是负数，意味着什么？**
   - 它们方向相反，越负相似度越「负相似」。RAG 检索时通常只取正相似度 top-k。
3. **`temperature = 0` 时 softmax 怎么算？**
   - 数学上除 0 爆炸，工程上等价于 argmax（贪心解码）。大多框架做了特殊处理。
4. **交叉熵为什么是 `-log(p_true)` 而不是别的？**
   - 信息论：`-log(p)` 是「事件发生时携带的信息量」。模型把真实事件预测得很可能（p 大）= 信息量少 = loss 小。

改一改 cell：把 `true_label` 改成 0（最不可能的类），看 loss 变多大。

## 自检 ✅

- [ ] 默写点积公式，并解释「为什么 attention score 用点积」。
- [ ] 默写余弦相似度公式，并解释「为什么 RAG 常用它」。
- [ ] 不查文档实现 softmax（含数值稳定）。
- [ ] 解释「温度对 softmax 的影响」，画一张草图。
- [ ] 解释「交叉熵 loss 为 0 意味着什么」。

## 下一步

→ [`05_transformer_mental_model.ipynb`](05_transformer_mental_model.ipynb)